# 06 - Prepare Fine-tuning Dataset No Leakage

Creates instruction-tuning train/validation JSONL files from auxiliary QA records only. The locked benchmark is used only for leakage filtering and is never included in training.

In [ ]:
from pathlib import Path
import json
import sys
import importlib

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ModuleNotFoundError:
    print('Not running in Google Colab; using local filesystem paths.')

DRIVE_ROOT = Path('/content/drive/MyDrive/rag')
sys.path = [str(DRIVE_ROOT)] + [p for p in sys.path if p != str(DRIVE_ROOT)]
for name in list(sys.modules):
    if name == 'src' or name.startswith('src.'):
        del sys.modules[name]
importlib.invalidate_caches()

config = json.loads((DRIVE_ROOT / 'project_config.json').read_text(encoding='utf-8'))
qa_csv = DRIVE_ROOT / config['qa_auxiliary_csv']
benchmark_csv = DRIVE_ROOT / config['benchmark_csv']
train_jsonl = DRIVE_ROOT / 'data/processed/finetune_train.jsonl'
val_jsonl = DRIVE_ROOT / 'data/processed/finetune_val.jsonl'
leakage_report = DRIVE_ROOT / 'reports/leakage_report.json'

for path in [qa_csv, benchmark_csv]:
    if not path.exists():
        raise FileNotFoundError(path)

qa_csv, benchmark_csv, train_jsonl, val_jsonl, leakage_report

In [ ]:
from src.prepare_finetune_dataset import prepare_finetune_dataset

report = prepare_finetune_dataset(
    qa_auxiliary_csv=qa_csv,
    benchmark_csv=benchmark_csv,
    output_train_jsonl=train_jsonl,
    output_val_jsonl=val_jsonl,
    leakage_report_json=leakage_report,
    val_ratio=0.1,
    seed=42,
    similarity_threshold=0.88,
    max_samples=None,
)

report

In [ ]:
import json

for path in [train_jsonl, val_jsonl]:
    print(path, path.exists(), round(path.stat().st_size / 1024, 1), 'KB')
    with path.open('r', encoding='utf-8') as f:
        first = json.loads(next(f))
    print(first.keys())
    print(first['instruction'][:120])
    print(first['input'][:200])
    print(first['output'][:200])
    print('---')

Expected outputs:

- `data/processed/finetune_train.jsonl`
- `data/processed/finetune_val.jsonl`
- `reports/leakage_report.json`

Next notebook will use the same base LLM checkpoint for LoRA/QLoRA fine-tuning.